# Self-Supervised Learning: SimCLR pada CIFAR-10 (PyTorch)

**Buku:** *Deep Learning dengan Python* — Bab 12, Self-Supervised Learning

Encoder ResNet-18 dilatih **tanpa satu label pun** dengan tujuan kontrastif: dua augmentasi dari gambar yang sama harus berdekatan, sedangkan gambar lain harus berjauhan. Kualitas representasinya diukur dengan *k-NN*, *linear probe*, dan skenario sedikit label.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Notebook ini memakai **PyTorch** murni (sesuai kerangka kode Bab 12–13).

**Perkiraan waktu (GPU T4):** ±60 menit untuk 50 epoch

**Isi notebook:**
- Augmentasi SimCLR: *random resized crop*, *color jitter*, *grayscale*.
- Loss NT-Xent (InfoNCE) yang ditulis dari nol.
- Pemantauan dengan *k-NN* berbobot (tanpa melatih pengklasifikasi).
- *Linear probe* dengan 1%, 10%, dan 100% label, dibandingkan dengan pelatihan terawasi dari nol.
- Visualisasi t-SNE ruang representasi.

In [ ]:
import torch, torchvision, numpy as np
perangkat = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__, "| perangkat:", perangkat)

In [ ]:
import time, math
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms as T
import matplotlib.pyplot as plt

EPOCH = 50            # SimCLR asli: 500-1000 epoch; 50 sudah cukup untuk melihat polanya
BATCH = 512
SUHU = 0.5

## 1. Augmentasi: dua pandangan dari satu gambar

In [ ]:
pakai_amp = perangkat == "cuda"
def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=pakai_amp)

RERATA, SIMPANGAN = (0.4914, 0.4822, 0.4465), (0.2471, 0.2435, 0.2616)
aug = T.Compose([T.RandomResizedCrop(32, scale=(0.2, 1.0)),
                 T.RandomHorizontalFlip(),
                 T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
                 T.RandomGrayscale(p=0.2),
                 T.ToTensor(), T.Normalize(RERATA, SIMPANGAN)])
polos = T.Compose([T.ToTensor(), T.Normalize(RERATA, SIMPANGAN)])

class DuaPandangan:
    def __init__(self, t):
        self.t = t
    def __call__(self, x):
        return self.t(x), self.t(x)

ds_pretrain = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=DuaPandangan(aug))
ds_latih = torchvision.datasets.CIFAR10("data", train=True, transform=polos)
ds_uji = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=polos)
loader_pretrain = DataLoader(ds_pretrain, batch_size=BATCH, shuffle=True, drop_last=True,
                             num_workers=2, pin_memory=True)

def tampil(t):
    return (t.permute(1, 2, 0) * torch.tensor(SIMPANGAN) + torch.tensor(RERATA)).clamp(0, 1)

fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    (v1, v2), _ = ds_pretrain[i]
    ax[0, i].imshow(tampil(v1)); ax[1, i].imshow(tampil(v2))
    ax[0, i].axis("off"); ax[1, i].axis("off")
plt.suptitle("Pasangan positif: kolom yang sama = gambar asal yang sama"); plt.show()

## 2. Encoder, *projection head*, dan loss NT-Xent

In [ ]:
def resnet18_cifar():
    m = torchvision.models.resnet18(num_classes=10)
    m.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)   # gambar kecil: stem 3x3, tanpa max-pool
    m.maxpool = nn.Identity()
    m.fc = nn.Identity()                                # keluaran: fitur 512-dim
    return m

class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = resnet18_cifar()
        self.proyektor = nn.Sequential(nn.Linear(512, 512), nn.BatchNorm1d(512),
                                       nn.ReLU(inplace=True), nn.Linear(512, 128))
    def forward(self, x):
        return self.proyektor(self.encoder(x))

def nt_xent(z1, z2, suhu=SUHU):
    """Untuk setiap sampel, pasangannya adalah 'kelas benar' di antara 2N-1 kandidat."""
    z = F.normalize(torch.cat([z1, z2]).float(), dim=1)
    n = z1.size(0)
    sim = z @ z.T / suhu
    sim.fill_diagonal_(float("-inf"))                   # buang kemiripan dengan diri sendiri
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, target)

# Uji cepat: pasangan identik -> loss kecil; pasangan acak -> loss ~ log(2N-1)
z = torch.randn(8, 128)
print(f"identik: {nt_xent(z, z).item():.3f} | acak: {nt_xent(z, torch.randn(8, 128)).item():.3f} | log(15) = {math.log(15):.3f}")

## 3. Evaluasi representasi: *k-NN* dan *linear probe*

In [ ]:
@torch.no_grad()
def ekstrak(encoder, ds):
    encoder.eval()
    fitur, label = [], []
    for x, y in DataLoader(ds, batch_size=512, num_workers=2):
        with autocast():
            fitur.append(encoder(x.to(perangkat)).float())
        label.append(y.to(perangkat))
    encoder.train()
    return F.normalize(torch.cat(fitur), dim=1), torch.cat(label)

@torch.no_grad()
def akurasi_knn(f_latih, y_latih, f_uji, y_uji, k=200, suhu=0.1):
    benar = 0
    for i in range(0, len(f_uji), 1000):
        sim, idx = (f_uji[i:i + 1000] @ f_latih.T).topk(k, dim=1)
        suara = torch.zeros(sim.size(0), 10, device=sim.device)
        suara.scatter_add_(1, y_latih[idx], (sim / suhu).exp())
        benar += (suara.argmax(1) == y_uji[i:i + 1000]).sum().item()
    return benar / len(f_uji)

def linear_probe(f_latih, y_latih, f_uji, y_uji, epoch=100):
    clf = nn.Linear(f_latih.size(1), 10).to(perangkat)
    opt = torch.optim.Adam(clf.parameters(), lr=1e-3, weight_decay=1e-5)
    for _ in range(epoch):
        urutan = torch.randperm(len(f_latih), device=perangkat)
        for i in range(0, len(f_latih), 1024):
            j = urutan[i:i + 1024]
            loss = F.cross_entropy(clf(f_latih[j]), y_latih[j])
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (clf(f_uji).argmax(1) == y_uji).float().mean().item()

model = SimCLR().to(perangkat)
f_l, y_l = ekstrak(model.encoder, ds_latih); f_u, y_u = ekstrak(model.encoder, ds_uji)
print(f"Encoder ACAK (belum dilatih): k-NN = {akurasi_knn(f_l, y_l, f_u, y_u):.2%}")

## 4. Pretraining kontrastif

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
jadwal = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCH * len(loader_pretrain))
skala = torch.amp.GradScaler(enabled=pakai_amp)
riwayat = {"epoch": [], "loss": [], "knn": []}

for epoch in range(1, EPOCH + 1):
    t0, total = time.time(), 0.0
    for (v1, v2), _ in loader_pretrain:
        v1, v2 = v1.to(perangkat, non_blocking=True), v2.to(perangkat, non_blocking=True)
        with autocast():
            z1, z2 = model(v1), model(v2)
        loss = nt_xent(z1, z2)
        opt.zero_grad(set_to_none=True)
        skala.scale(loss).backward(); skala.step(opt); skala.update(); jadwal.step()
        total += loss.item()
    riwayat["epoch"].append(epoch); riwayat["loss"].append(total / len(loader_pretrain))
    pesan = f"Epoch {epoch:3d} | loss {riwayat['loss'][-1]:.4f} | {time.time() - t0:.0f} dtk"
    if epoch % 10 == 0 or epoch == EPOCH:
        f_l, y_l = ekstrak(model.encoder, ds_latih); f_u, y_u = ekstrak(model.encoder, ds_uji)
        riwayat["knn"].append((epoch, akurasi_knn(f_l, y_l, f_u, y_u)))
        pesan += f" | k-NN {riwayat['knn'][-1][1]:.2%}"
    print(pesan)

torch.save(model.encoder.state_dict(), "simclr_encoder.pt")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(riwayat["epoch"], riwayat["loss"]); ax[0].set(xlabel="Epoch", ylabel="Loss NT-Xent")
e, a = zip(*riwayat["knn"]); ax[1].plot(e, a, "o-"); ax[1].set(xlabel="Epoch", ylabel="Akurasi k-NN")
plt.show()

## 5. Seberapa berguna representasinya? Sedikit label vs dari nol

In [ ]:
LANGKAH_SUP = 2000
aug_sup = T.Compose([T.RandomCrop(32, padding=4, padding_mode="reflect"), T.RandomHorizontalFlip(),
                     T.ToTensor(), T.Normalize(RERATA, SIMPANGAN)])
ds_latih_aug = torchvision.datasets.CIFAR10("data", train=True, transform=aug_sup)

def terawasi_dari_nol(indeks, langkah=LANGKAH_SUP, batch=128):
    m = resnet18_cifar(); m.fc = nn.Linear(512, 10); m = m.to(perangkat)
    opt = torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4, nesterov=True)
    jdw = torch.optim.lr_scheduler.CosineAnnealingLR(opt, langkah)
    sampler = torch.utils.data.RandomSampler(Subset(ds_latih_aug, indeks), replacement=True,
                                             num_samples=batch * langkah)
    sk = torch.amp.GradScaler(enabled=pakai_amp)
    m.train()
    for x, y in DataLoader(Subset(ds_latih_aug, indeks), batch_size=batch, sampler=sampler, num_workers=2):
        with autocast():
            loss = F.cross_entropy(m(x.to(perangkat)), y.to(perangkat))
        opt.zero_grad(set_to_none=True); sk.scale(loss).backward(); sk.step(opt); sk.update(); jdw.step()
    m.eval(); benar = 0
    with torch.no_grad():
        for x, y in DataLoader(ds_uji, batch_size=512):
            benar += (m(x.to(perangkat)).argmax(1).cpu() == y).sum().item()
    return benar / len(ds_uji)

f_l, y_l = ekstrak(model.encoder, ds_latih); f_u, y_u = ekstrak(model.encoder, ds_uji)
rng = np.random.default_rng(0)
print(f"{'Label':>8s} | {'SimCLR + linear probe':>22s} | {'ResNet-18 dari nol':>18s}")
for persen in [1, 10, 100]:
    idx = np.sort(rng.choice(50000, 500 * persen, replace=False))
    a_probe = linear_probe(f_l[idx], y_l[idx], f_u, y_u)
    a_nol = terawasi_dari_nol(idx) if persen < 100 else float("nan")
    print(f"{persen:>7d}% | {a_probe:>22.2%} | {a_nol:>18.2%}")

Dengan 1% label (50 gambar per kelas), representasi SimCLR jauh mengungguli pelatihan dari nol. Selisih ini menyempit saat label bertambah. Inilah alasan *self-supervised pretraining* sangat berharga di domain dengan label mahal.

## 6. t-SNE ruang representasi

In [ ]:
from sklearn.manifold import TSNE

idx = np.random.default_rng(1).choice(10000, 3000, replace=False)
peta = TSNE(n_components=2, init="pca", random_state=0).fit_transform(f_u[idx].cpu().numpy())
plt.figure(figsize=(8, 7))
sebar = plt.scatter(peta[:, 0], peta[:, 1], c=y_u[idx].cpu().numpy(), cmap="tab10", s=4)
plt.legend(sebar.legend_elements()[0], ds_uji.classes, fontsize=8, markerscale=1.5)
plt.title("t-SNE fitur SimCLR (data uji, tanpa label saat pelatihan)"); plt.axis("off"); plt.show()

**Eksperimen:**
1. Hapus `ColorJitter` dari augmentasi. Akurasi *k-NN* akan turun tajam, karena model bisa "curang" memakai histogram warna.
2. Ubah `SUHU` menjadi 0,1 atau 1,0.
3. Lakukan *linear probe* pada keluaran `proyektor` (128-dim), bukan `encoder`. Mengapa hasilnya lebih buruk?